# Examen 3: Modelo de regresión

## Pregunta 1: Inferencia y Diagnóstico Econométrico

## Pregunta 2: Modelado Predictivo e Ingeniería de Características (50 %)

In [1]:
#Cargar librerías y datos:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Lectura del dataframe
df = pd.read_csv("data/raw/consumo_energia.csv")

1. **Ingeniería de Características y Filtrado con LASSO:** A partir de las variables originales de la base de datos, genere todas las interacciones y términos polinómicos de grado 2 posibles. Posteriormente, estandarice esta nueva matriz de datos y aplique un modelo de regularización LASSO (con validación cruzada) exclusivamente como método de selección de variables.

In [3]:
# Inspeccionar las primeras filas para confirmar nombres de columnas
print(f"Dimensiones orifinales: {df.shape}")
print("Columnas originales:")
print(df.columns.tolist())
print("-" * 50)

# 1. Efecto No Lineal: Cuadrado de la temperatura
df['temperatura_media_sq'] = df['temperatura_media'] ** 2

# 2. Interacción Lógica: Temperatura x Tipo de Edificio
# Primero, convertimos variables categóricas a dummies (One-Hot Encoding)
# Observo que también está 'dia_semana', la convertiremos también de una vez.
df_procesado = pd.get_dummies(df, columns=['tipo_edificio', 'dia_semana'], drop_first=True, dtype=int)

# Identificamos las columnas dummy del tipo de edificio
dummy_edificios = [col for col in df_procesado.columns if col.startswith('tipo_edificio_')]

# Creamos las interacciones
for col in dummy_edificios:
    interaction_col_name = f"interaccion_temp_x_{col}"
    df_procesado[interaction_col_name] = df_procesado['temperatura_media'] * df_procesado[col]

print(f"Dimensiones de la nueva matriz de datos: {df_procesado.shape}")
print("\nNuevas columnas generadas:")
print(df_procesado.columns.tolist())

Dimensiones orifinales: (800, 7)
Columnas originales:
['temperatura_media', 'visitantes_diarios', 'tipo_edificio', 'humedad_relativa', 'velocidad_viento', 'dia_semana', 'consumo_kwh']
--------------------------------------------------
Dimensiones de la nueva matriz de datos: (800, 14)

Nuevas columnas generadas:
['temperatura_media', 'visitantes_diarios', 'humedad_relativa', 'velocidad_viento', 'consumo_kwh', 'temperatura_media_sq', 'tipo_edificio_Corporativo', 'dia_semana_Jueves', 'dia_semana_Lunes', 'dia_semana_Martes', 'dia_semana_Miercoles', 'dia_semana_Sabado', 'dia_semana_Viernes', 'interaccion_temp_x_tipo_edificio_Corporativo']


- ¿Cuántas variables generó en total y cuántas sobrevivieron al filtro de LASSO (coeficiente distinto de cero)?


Teníamos originalmente columnas como temperatura_media, tipo_edificio y dia_semana. Ahora, tras aplicar la Ingeniería de Características (efectos no lineales, codificación One-Hot y términos de interacción), nuestra matriz X ha pasado a tener 13 predictores

Las nuevas variables clave que logramos construir son:

* **temperatura_media_sq:** Captura la concavidad (parábola) del consumo de energía.

* **tipo_edificio_Corporativo:** Nuestra dummy. La categoría base elegida automáticamente fue "Comercial".

* **interaccion_temp_x_tipo_edificio_Corporativo:** La variable de "sinergia".

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

# 1. Separamos nuestra matriz de características X y el vector objetivo y
# Omitimos la columna objetivo 'consumo_kwh' de X
X = df_procesado.drop(columns=['consumo_kwh'])
y = df_procesado['consumo_kwh']

# 2. Particionamiento de los datos (80% Entrenamiento, 20% Prueba)
# Fijamos un random_state para reproducibilidad académica
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Protocolo de Aislamiento: Estandarización Z-Score
scaler = StandardScaler()

# Ajustamos (aprendemos media y varianza) SOLO en Train y transformamos
X_train_scaled = scaler.fit_transform(X_train)

# Transformamos el Test usando la regla aprendida en Train (evita fuga de datos)
X_test_scaled = scaler.transform(X_test)

# 4. Ajuste del Modelo LASSO
# El hiperparámetro alpha representa nuestro lambda en la fórmula. 
# Un alpha muy alto apagará casi todas las variables. Un alpha muy bajo lo hará igual a OLS.
# Para iniciar, probaremos con alpha = 50.0
lasso_model = Lasso(alpha=50.0, random_state=42)

# Optimizamos la función de costo analizada previamente
lasso_model.fit(X_train_scaled, y_train)

# 5. Extraemos los coeficientes para ver cuáles "sobrevivieron" al diamante de LASSO
coeficientes_lasso = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente_Lasso': lasso_model.coef_
})

# Filtramos las variables descartadas (las que LASSO volvió estrictamente 0)
variables_sobrevivientes = coeficientes_lasso[coeficientes_lasso['Coeficiente_Lasso'] != 0].copy()

# Ordenamos por valor absoluto para ver las de mayor impacto (como pide el examen)
variables_sobrevivientes['Magnitud_Absoluta'] = np.abs(variables_sobrevivientes['Coeficiente_Lasso'])
variables_sobrevivientes = variables_sobrevivientes.sort_values(by='Magnitud_Absoluta', ascending=False)

print("Variables que sobrevivieron al filtro LASSO:")
print(variables_sobrevivientes[['Variable', 'Coeficiente_Lasso']])

Variables que sobrevivieron al filtro LASSO:
                    Variable  Coeficiente_Lasso
4       temperatura_media_sq        3798.991540
0          temperatura_media       -3396.908655
5  tipo_edificio_Corporativo        1287.759979
2           humedad_relativa          34.921968
7           dia_semana_Lunes           3.756172


- Indique cuáles son las tres (3) variables más importantes descubiertas por LASSO (las de mayor coeficiente en valor absoluto).

De las 13 variables originales que construimos y le pasamos al modelo, la penalización L1​ "apagó" 8 de ellas, dejando su coeficiente exactamente en cero. Esto significa que el modelo consideró que interacciones y otros días de la semana aportaban más ruido que señal.

Las 5 variables sobrevivientes son:

* temperatura_media_sq (La curvatura)

* temperatura_media (La tendencia lineal climática)

* tipo_edificio_Corporativo (El salto por tipo de edificio)

* humedad_relativa

* dia_semana_Lunes

Selección de Variables con LASSO ($L_1$) y Análisis de Signos

Para evitar el sobreajuste y filtrar las variables irrelevantes, aplicamos la regresión LASSO, que modifica la función de pérdida agregando una penalización basada en la norma $L_1$:

$$J(\theta) = \frac{1}{2m} \|X\theta - y\|^2_2 + \lambda \sum_{j=1}^{d} |\theta_j|$$

La geometría del espacio de parámetros en LASSO (un rombo) fuerza a que los coeficientes de las variables que no aportan suficiente "señal" colapsen exactamente a cero. 

**Análisis del Top 3 de Variables Sobrevivientes:**
Al ordenar los coeficientes por magnitud (valor absoluto $|\theta_j|$), encontramos:
1. `temperatura_media_sq`: $+3798.99$
2. `temperatura_media`: $-3396.91$
3. `tipo_edificio_Corporativo`: $+1287.76$



2. **Modelo Post-LASSO y Evaluación del Negocio:** Utilizando ´unicamente las variables que sobrevivieron al filtro de LASSO en el literal anterior, ajuste un modelo de Regresión Lineal Clásica (OLS sin penalización). Evalue el poder predictivo de este modelo en su conjunto de prueba calculando el R2, la Raíz del Error Cuadrático Medio (RMSE) y el Error Absoluto Medio (MAE).

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Filtramos las matrices de entrenamiento y prueba SOLO con las variables que sobrevivieron a LASSO
# (Asumimos que 'variables_sobrevivientes' es el DataFrame que creamos en el paso anterior)
columnas_seleccionadas = variables_sobrevivientes['Variable'].tolist()

X_train_post = X_train_scaled[:, X.columns.isin(columnas_seleccionadas)]
X_test_post = X_test_scaled[:, X.columns.isin(columnas_seleccionadas)]

# 2. Entrenamos el modelo Post-LASSO (OLS sin penalización)
ols_model = LinearRegression()
ols_model.fit(X_train_post, y_train) # Aquí ocurre la magia de las ecuaciones normales

# 3. Generamos las predicciones sobre el conjunto de prueba aislado
y_pred = ols_model.predict(X_test_post)

# 4. Calculamos las métricas de error matricialmente
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Raíz del MSE
mae = mean_absolute_error(y_test, y_pred)

print(f"--- Evaluación del Modelo Post-LASSO ---")
print(f"R² (Poder explicativo): {r2:.4f}")
print(f"RMSE (Penaliza outliers): {rmse:.2f} kWh")
print(f"MAE (Error promedio): {mae:.2f} kWh")

--- Evaluación del Modelo Post-LASSO ---
R² (Poder explicativo): 0.9447
RMSE (Penaliza outliers): 445.35 kWh
MAE (Error promedio): 378.81 kWh


- **Segun el $R^2$** Nuestro modelo es capaz de explicar el 94.47% de toda la variabilidad del consumo eléctrico diario de los edificios. Es un valor excepcionalmente alto. Significa que el ruido representa apenas el 5.53% del comportamiento de los datos. La combinación de temperatura, tipo de edificio, humedad y ser lunes es extremadamente poderosa para dictar la demanda energética.

- **El Error Absoluto Medio (MAE = 378.81 kWh)**
En un día cualquiera, nuestra predicción de consumo eléctrico se equivoca, en promedio, por 378.81 kWh (ya sea por encima o por debajo del valor real). Si consideramos que en nuestro dataset los consumos rondan entre los 4,000 y 12,000 kWh aproximadamente, un error promedio de ≈378 kWh representa un margen de equivocación muy bajo (menos del 10% relativo en la mayoría de los casos)

- **La Raíz del Error Cuadrático Medio (RMSE = 445.35 kWh)**
Aquí está la verdadera "prueba de fuego" diagnóstica. El RMSE aplica una penalización cuadrática, lo que significa que amplifica agresivamente los errores grandes antes de promediarlos.

**El análisis comparativo (RMSE vs. MAE)**
En nuestro caso, 445.35 kWh vs. 378.81 kWh es una diferencia bastante razonable. Esto nos confirma estadísticamente que nuestro modelo no está sufriendo de fallas catastróficas ni sobreajuste salvaje en el conjunto de prueba. Los errores están distribuidos de manera relativamente uniforme, sin colas excesivamente pesadas.


3. **El Duelo Final (El Principio de Parsimonia):** El equipo de Ingeniería de TI de la empresa le advierte que llevar a producción un modelo con decenas de variables es costoso y difícil de mantener. Construya un modelo “parsimonioso” (ligero) utilizando exclusivamente las 3 variables más importantes que identificó en el literal anterior

**Respuesta:**
Atendiendo a las restricciones de TI, planteamos un modelo parsimonioso restringido al espacio de  dimensión $d=3$. La ecuación estructural de nuestro modelo a desplegar en producción será:

$$\hat{y} = \hat{\beta}_0 + \hat{\beta}_1 (\text{temp\_sq}) + \hat{\beta}_2 (\text{temp}) + \hat{\beta}_3 (\text{edificio\_Corp})$$

Calcularemos el $RMSE_{ligero}$ para este nuevo modelo y lo compararemos contra el $RMSE_{completo}$ (el de 5 variables), evaluando si el sacrificio en dimensionalidad (reducción de costos de cómputo y mantenimiento) se justifica frente a la posible pérdida de precisión estadística.

In [8]:
# 1. Definimos las 3 variables estrella
top_3_columnas = [
    'temperatura_media_sq', 
    'temperatura_media', 
    'tipo_edificio_Corporativo'
]

# 2. Filtramos nuestras matrices ya escaladas para quedarnos solo con el Top 3
X_train_top3 = X_train_scaled[:, X.columns.isin(top_3_columnas)]
X_test_top3 = X_test_scaled[:, X.columns.isin(top_3_columnas)]

# 3. Entrenamos el Modelo Ligero (OLS)
ols_ligero = LinearRegression()
ols_ligero.fit(X_train_top3, y_train)

# 4. Predecimos y evaluamos
y_pred_top3 = ols_ligero.predict(X_test_top3)
rmse_top3 = np.sqrt(mean_squared_error(y_test, y_pred_top3))

print(f"--- Duelo de Modelos en Datos de Prueba ---")
print(f"RMSE Modelo Completo (5 vars): {rmse:.2f} kWh")
print(f"RMSE Modelo Ligero (3 vars):   {rmse_top3:.2f} kWh")

--- Duelo de Modelos en Datos de Prueba ---
RMSE Modelo Completo (5 vars): 445.35 kWh
RMSE Modelo Ligero (3 vars):   441.99 kWh


Se nota un menor valor RMSE **con tres variables**, lo que indica una mejor precisión del modelo. Agregar más variables no siempre mejora el modelo. De hecho, a menudo lo empeora en la fase de prueba.

<span style="color:red">**Conclusión y Recomendación a la Gerencia (Torres Milenio S.A.)**</span>

**Comparativa de Modelos:**
* $RMSE_{completo}$ (5 variables): 445.35 kWh
* $RMSE_{ligero}$ (3 variables): 441.99 kWh

**Recomendación de Negocio:**
Como Analista de Datos Senior, **recomiendo contundentemente la implementación en producción del Modelo Ligero (Parsimonioso) de 3 variables.** **Justificación Técnica y de Negocio:**
1. **Precisión Estadística Superadora:** Contraintuitivamente, el modelo de 3 variables no solo no perdió precisión, sino que superó al modelo de 5 variables (reduciendo el error cuadrático medio en los datos de prueba). Esto demuestra que las variables de humedad y día de la semana estaban introduciendo ruido y ligero sobreajuste (overfitting).
2. **El Principio de Parsimonia (Navaja de Ockham):** Ante dos modelos con desempeño similar, siempre se debe elegir el más simple. 
3. **Simplicidad y Reducción de Costos IT:** Un modelo que requiere recolectar, limpiar y procesar solo 2 variables originales (Temperatura y Tipo de Edificio) es computacionalmente más económico, reduce la latencia en tiempo real, minimiza los puntos de fallo en la ingesta de datos y facilita la mantenibilidad a largo plazo por parte del equipo de Ingeniería de TI.

In [ ]:
# Es importante identificar cuáles variables son continuas (float64) y cuáles categóricas (object)
display(df.info())

In [ ]:
# Configuración de estilo académico para las gráficas
sns.set_context("paper", font_scale=1.2)
plt.figure(figsize=(10, 6))

# 2. Análisis del vector objetivo (y)
# Evaluamos la distribución del consumo de energía. ¿Tiene asimetría positiva?
sns.histplot(df['consumo_kwh'], kde=True, color='blue')
plt.title('Distribución de la Variable Objetivo: Consumo de Energía (kWh)')
plt.xlabel('Consumo (kWh)')
plt.ylabel('Frecuencia')
plt.show()

## Pregunta 3: Modelado Predictivo, LASSO y Principio de Parsimonia